In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import geopandas as gpd
import planetary_computer as pc
import pystac_client
import pandas as pd

CONFIG_DIR = project_root / "config"
RAW_DIR = project_root / "data" / "raw" / "sentinel-2"

RAW_DIR.mkdir(parents=True, exist_ok=True)

STUDY_AREA_PATH = CONFIG_DIR / "study_area.geojson"
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SENTINEL2_COLLECTION = "sentinel-2-l2a"

print("Project root:", project_root)
print("Study area:", STUDY_AREA_PATH)
print("Raw data folder:", RAW_DIR)

Project root: c:\Users\pc\Documents\Data Science Projects\Shoremonitor
Study area: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\config\study_area.geojson
Raw data folder: c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\raw\sentinel-2


In [6]:
import geopandas as gpd

study_area = gpd.read_file("../config/study_area.geojson")

print(study_area)
print("\nCRS:", study_area.crs)
print("\nBounds:", study_area.total_bounds)

                           name country    analysis_level  \
0  Mombasa-Nyali-Bamburi-Shanzu   Kenya  detailed_hotspot   

                                            geometry  
0  POLYGON ((39.65 -4.1, 39.85 -4.1, 39.85 -3.9, ...  

CRS: EPSG:4326

Bounds: [39.65 -4.1  39.85 -3.9 ]


In [7]:
import planetary_computer as pc
import pystac_client

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SENTINEL2_COLLECTION = "sentinel-2-l2a"

catalog = pystac_client.Client.open(
    STAC_URL,
    modifier=pc.sign_inplace,
)

print("Connected to Planetary Computer STAC catalog")
print("Collection:", SENTINEL2_COLLECTION)

Connected to Planetary Computer STAC catalog
Collection: sentinel-2-l2a


In [8]:
# Convert the study area to WGS84 and calculate its bounding box
study_area = gpd.read_file(STUDY_AREA_PATH)
if study_area.crs is None or study_area.crs.to_epsg() != 4326:
    study_area = study_area.to_crs(epsg=4326)

study_area_bounds = study_area.total_bounds

# total_bounds returns [minx, miny, maxx, maxy]
bbox = [
    study_area_bounds[0],
    study_area_bounds[1],
    study_area_bounds[2],
    study_area_bounds[3],
]

print("Study-area bounding box:")
print(bbox)

Study-area bounding box:
[np.float64(39.65), np.float64(-4.1), np.float64(39.85), np.float64(-3.9)]


In [9]:
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
MAX_CLOUD_COVER = 20

search = catalog.search(
    collections=[SENTINEL2_COLLECTION],
    bbox=bbox,
    datetime=f"{START_DATE}/{END_DATE}",
    query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
)

items = list(search.items())

print(f"Found {len(items)} Sentinel-2 L2A scenes")

Found 8 Sentinel-2 L2A scenes


In [10]:
scene_records = []

for item in items:
    scene_records.append(
        {
            "scene_id": item.id,
            "datetime": item.datetime,
            "cloud_cover": item.properties.get("eo:cloud_cover"),
            "mgrs_tile": item.properties.get("s2:mgrs_tile"),
            "platform": item.properties.get("platform"),
            "collection": item.collection_id,
        }
    )

scenes = pd.DataFrame(scene_records)
scenes = scenes.sort_values("datetime")

scenes

,scene_id,datetime,cloud_cover,mgrs_tile,platform,collection
6,S2A_MSIL2A_20240310T072721_R049_T37MER_2024031...,2024-03-10 07:27:21.024+00:00,19.834769,37MER,Sentinel-2A,sentinel-2-l2a
7,S2A_MSIL2A_20240310T072721_R049_T37MER_2024031...,2024-03-10 07:27:21.024+00:00,19.834769,37MER,Sentinel-2A,sentinel-2-l2a
5,S2B_MSIL2A_20240325T072619_R049_T37MER_2024032...,2024-03-25 07:26:19.024+00:00,9.632075,37MER,Sentinel-2B,sentinel-2-l2a
4,S2A_MSIL2A_20240529T074541_R049_T37MER_2024052...,2024-05-29 07:45:41.024+00:00,0.122262,37MER,Sentinel-2A,sentinel-2-l2a
3,S2B_MSIL2A_20240613T072619_R049_T37MER_2024061...,2024-06-13 07:26:19.024+00:00,17.834778,37MER,Sentinel-2B,sentinel-2-l2a
2,S2B_MSIL2A_20240812T072619_R049_T37MER_2024081...,2024-08-12 07:26:19.024+00:00,18.206960,37MER,Sentinel-2B,sentinel-2-l2a
1,S2A_MSIL2A_20241016T072901_R049_T37MER_2024101...,2024-10-16 07:29:01.024+00:00,12.276568,37MER,Sentinel-2A,sentinel-2-l2a
0,S2A_MSIL2A_20241105T073101_R049_T37MER_2024110...,2024-11-05 07:31:01.024+00:00,17.766325,37MER,Sentinel-2A,sentinel-2-l2a


In [11]:
first_item = items[0]

print("Scene ID:", first_item.id)
print("Date:", first_item.datetime)
print("\nAvailable assets:")

for asset_key in sorted(first_item.assets.keys()):
    print(asset_key)

Scene ID: S2A_MSIL2A_20241105T073101_R049_T37MER_20241105T110448
Date: 2024-11-05 07:31:01.024000+00:00

Available assets:
AOT
B01
B02
B03
B04
B05
B06
B07
B08
B09
B11
B12
B8A
SCL
WVP
datastrip-metadata
granule-metadata
inspire-metadata
product-metadata
rendered_preview
safe-manifest
tilejson
visual


In [12]:
metadata_path = RAW_DIR / "sentinel2_scene_catalog_2024.csv"
scenes.to_csv(metadata_path, index=False)

print(f"Saved {len(scenes)} scene records to:")
print(metadata_path)

Saved 8 scene records to:
c:\Users\pc\Documents\Data Science Projects\Shoremonitor\data\raw\sentinel-2\sentinel2_scene_catalog_2024.csv
